In [20]:
import numpy as np 
import matplotlib.pyplot as plt 
import os 
import getpass 
from langchain_groq import ChatGroq
from langchain_aws import ChatBedrock
# Libraries and codes to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

In [2]:
def set_if_undefined(var:str):  
    if os.environ.get(var):  
        return 
    os.environ[var] = getpass.getpass(var)  
set_if_undefined("GROQ_API_KEY")    

GROQ_API_KEY ········


In [3]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/1r_mM6ZPYNxcFv65QkzubA/California-Culinary-Map.txt

--2026-06-04 17:55:35--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/1r_mM6ZPYNxcFv65QkzubA/California-Culinary-Map.txt
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 198.23.119.245
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|198.23.119.245|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 74679 (73K) [text/plain]
Saving to: ‘California-Culinary-Map.txt.1’

California-Culinary 100%[===================>]  72.93K  --.-KB/s    in 0.1s    

2026-06-04 17:55:36 (633 KB/s) - ‘California-Culinary-Map.txt.1’ saved [74679/74679]



In [4]:
### Your Code Here:
### 1.1: Define the file_path to the text file
file_path = "California-Culinary-Map.txt"

### 1.2: Open the text file
with open(file_path, "r")as f: 
    content = f.read()

### 1.3: Print the first 100 characters of the restaurant data
content[:500]

'### The Culinary Map of California\n\n**The Gilded Artichoke** brings a **bohemian chic** energy to the hills of **Silver Lake**, operating as an **upscale bistro** that prioritizes **Farm-to-Table Californian** ingredients. The space feels like a high-end greenhouse with its reclaimed wood and floor-to-ceiling windows, perfectly complementing the **4.5/5** rating earned by its lavender-rubbed roasted chicken and delicate heirloom tomato tarts.  Price range: $$$$\n\nDown in **Santa Monica**, **Mar d'

In [5]:
### Your Code Here:
### 2.1: Split the restaurant paragraphs into list (hint: use .split)
restaurant_list = content.split("\n\n")

### 2.2: Since the first item is the dataset name, we remove it
restaurant_list = restaurant_list[1:]

### 2.3: Print out the number of restaurants we have
print(len(restaurant_list))

### 2.4: Print out the first item to have a closer look at the content
print(restaurant_list[0])

210
**The Gilded Artichoke** brings a **bohemian chic** energy to the hills of **Silver Lake**, operating as an **upscale bistro** that prioritizes **Farm-to-Table Californian** ingredients. The space feels like a high-end greenhouse with its reclaimed wood and floor-to-ceiling windows, perfectly complementing the **4.5/5** rating earned by its lavender-rubbed roasted chicken and delicate heirloom tomato tarts.  Price range: $$$$


In [21]:
def llm_model(system_msg, prompt_txt):
    #system_msg: the system message given to the LLM
    #prompt_txt: the user prompt
    
    
    ### 1.1: Define the model by ModelInference
    llm = ChatBedrock(
        model_id="amazon.nova-micro-v1:0",  # or "amazon.nova-micro-v1:0" (cheapest)
        region_name="us-east-1",
    )
    ### 1.2: Define the messages
    messages = [
        {"role": "system", "content": system_msg},
        {"role" : "user", "content": prompt_txt}
    ]

    ### 1.3: Get the final response output and return it
    response = llm.invoke(messages)

    return response.content

In [17]:
import time

def safe_llm_call(system_msg, prompt_txt, retries=3):
    for i in range(retries):
        try:
            return llm_model(system_msg, prompt_txt)
        except Exception:
            time.sleep(2)
    return "Failed after retries"

In [22]:
system_msg = "You are a helpful assistant."
prompt_txt = "Which place is warmer in winter? Hawaii or Greenland?"
print(safe_llm_call(system_msg, prompt_txt))

In winter, Hawaii is significantly warmer than Greenland. 

Hawaii, located in the middle of the Pacific Ocean, enjoys a tropical climate year-round. In the winter months, temperatures typically range from the lower 60s to mid-70s Fahrenheit (around 16 to 24 degrees Celsius) both during the day and night.

On the other hand, Greenland, which is an autonomous territory within the Kingdom of Denmark located in North America, has a polar climate. During the winter months, temperatures can plummet to well below freezing, often ranging from around 0 to 32 degrees Fahrenheit (-18 to 0 degrees Celsius) in many regions, and can be much colder in some interior areas.

Thus, Hawaii is much warmer compared to Greenland in the winter.


#Exercise 3: Prompt Engineering

In [9]:
EXAMPLE_RESTAURANT_PARAGRAPH = restaurant_list[1] #use the second restaurant paragraph as the example
EXAMPLE_OUTPUT = """
    {{
    "name": "Mar de Cortez",
    "location": "Santa Monica",
    "type": "casual taqueria",
    "food_style": "Baja-style seafood",
    "rating": 4.2,
    "price_range": 1,
    "signatures": [
        "beer-battered snapper tacos",
        "zesty octopus ceviche"
    ],
    "vibe": "salt-air energy",
    "environment": "a premier sun-drenched spot for open-air dining near the pier."
    "shortcomings": []
    }}
"""
### Design your prompt here
def restaurant_data_structure_prompt_generation(restaurant_paragraph):
    base_system_msg = f"""
    You are a data extraction assistant that converts restaurant descriptions into structured JSON format.
    Follow these rules strictly:
    - Extract only information explicitly mentioned in the description.
    - For price_range, convert dollar signs to an integer (e.g., $ = 1, $$ = 2, $$$ = 3, $$$$ = 4).
    - For shortcomings, extract any negatives or complaints mentioned, otherwise return an empty list [].
    - Return ONLY valid JSON with no extra text, explanation, or markdown backticks.
    """
    
    base_user_prompt = f"""
    Task:
    Convert the restaurant description below into a structured JSON object with the following fields:
    name, location, type, food_style, rating, price_range, signatures, vibe, environment, shortcomings.
    
    Restaurant description:
    {restaurant_paragraph}
    
    Example:
    Input Restaurant Description: {EXAMPLE_RESTAURANT_PARAGRAPH}
    Output:
    {EXAMPLE_OUTPUT}
    
    Now extract the JSON for the restaurant description above. Return only the JSON object.
    """
    
    return base_system_msg, base_user_prompt

In [23]:
# Unit test:
restaurant_paragraph = restaurant_list[0]
base_system_msg, base_user_prompt = restaurant_data_structure_prompt_generation(restaurant_paragraph=restaurant_paragraph)

test_response = llm_model(system_msg=base_system_msg, prompt_txt=base_user_prompt)
print(test_response)

{
    "name": "The Gilded Artichoke",
    "location": "Silver Lake",
    "type": "upscale bistro",
    "food_style": "Farm-to-Table Californian",
    "rating": 4.5,
    "price_range": 3,
    "signatures": [
        "lavender-rubbed roasted chicken",
        "delicate heirloom tomato tarts"
    ],
    "vibe": "bohemian chic",
    "environment": "high-end greenhouse with reclaimed wood and floor-to-ceiling windows",
    "shortcomings": []
}


In [24]:
# Validation
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional

### 3.1. Define the schema
class Restaurant(BaseModel):
    name: str
    location: str
    type: str
    food_style: str
    rating: Optional[float] = None
    price_range: Optional[int] = None
    signatures: List[str] = Field(default_factory=list)
    vibe: Optional[str] = None
    environment: str
    shortcomings: List[str] = Field(default_factory=list)


### 3.2. Use the validation method to validate the test_response from the unit test
try:
    restaurant_data = Restaurant.model_validate_json(test_response)
    print(f"Success! Validated: {restaurant_data.name}")
except ValidationError as e:
    print(f"Validation failed: {e.json()}")

Success! Validated: The Gilded Artichoke


Exercise 4: Structure all the restaurant data

In [12]:
def JSON_auto_repair_prompts(candidate_json_output, error_message):
    auto_repair_system_msg = """
    You are a JSON repair expert. Your sole responsibility is to fix invalid or malformed JSON outputs.
    Follow these rules strictly:
    - Return ONLY the corrected valid JSON object, no explanations or markdown backticks.
    - Do not change or remove any correct data, only fix what is broken.
    - Ensure the output is parseable by Python's json.loads() function.
    - If a field is missing, infer it from context or use a sensible default (e.g., [] for lists, null for unknown values).
    - For price_range, ensure it is an integer (e.g., $ = 1, $$ = 2, $$$ = 3, $$$$ = 4).
    """
    
    auto_repair_prompt = f"""
    The following JSON output is invalid or does not conform to the required schema.
    
    Incorrect JSON output:
    {candidate_json_output}
    
    Error message from schema validation:
    {error_message}
    
    Please fix the JSON output based on the error message above.
    Return ONLY the corrected valid JSON object.
    """
    
    return auto_repair_system_msg, auto_repair_prompt

In [25]:
structured_restaurant_lists = []
for i, restaurant_paragraph in enumerate(restaurant_list):
    ### 2.1: Produce your initial output
    system_msg, user_prompt = restaurant_data_structure_prompt_generation(restaurant_paragraph)
    response = llm_model(system_msg, user_prompt)
    ### 2.2: Validation and Auto Correction loop
    for _ in range(3):  # try up to 3 times
        try:
            parsed = json.loads(response)
            Restaurant.model_validate(parsed)
            break
        except (json.JSONDecodeError, ValidationError) as e:
            error_message = str(e)
            repair_system_msg, repair_prompt = JSON_auto_repair_prompts(response, error_message)
            response = llm_model(repair_system_msg, repair_prompt)
    ### 2.3: Append your finalized response to the structured_restaurant_lists
    structured_restaurant_lists.append(parsed)

    # A manual progress bar
    if (i+1) % 20 == 0:
        print(f'{i+1} out of {len(restaurant_list)} is done')
print('ALL DONE!!')

20 out of 210 is done
40 out of 210 is done
60 out of 210 is done
80 out of 210 is done
100 out of 210 is done
120 out of 210 is done
140 out of 210 is done
160 out of 210 is done
180 out of 210 is done
200 out of 210 is done
ALL DONE!!


In [26]:
### Your Code Here:
### Print the 50th item in the structured_restaurant_lists
structured_restaurant_lists[49]

{'name': 'The Neon Noodle Bar',
 'location': 'Monterey Park',
 'type': 'Hong Kong cafe',
 'food_style': 'instant noodle gourmet bowls, pineapple buns',
 'rating': 4.0,
 'price_range': 2,
 'signatures': ['instant noodle gourmet bowls', 'pineapple buns'],
 'vibe': 'frenetic and delicious energy of a late-night HK diner',
 'environment': 'high-energy',
 'shortcomings': []}

In [28]:
# Remove json.loads() - data is already parsed
structured_restaurant_lists_json = structured_restaurant_lists

# For each item in the restaurant list, assign it with an itemId to be consistent with the one in the user review data:
for i, response in enumerate(structured_restaurant_lists_json):
    response['itemId'] = 1000001 + i
    structured_restaurant_lists_json[i] = response

filename = 'structured_restaurant_data.json'
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(structured_restaurant_lists_json, f, indent=4)